La cohorte de modélisation comprend une seule ligne résidentielle par groupe candidat et écarte les groupes avec d’autres locaux bâtis, sauf dépendances. Les terrains peuvent toutefois partager le prix.

Le script `ml/training/train_baseline.py` compare deux références médianes, un Ridge et un Histogram Gradient Boosting. Les deux modèles apprennent sur janvier–septembre 2025 et sont évalués sur octobre–décembre 2025. `sale_date` sert au découpage et n’est pas donnée aux modèles.

La cible est transformée par `log1p`, puis reconvertie en euros. Les deux modèles utilisent les mêmes variables. Ridge encode les catégories en indicateurs (one-hot) ; le modèle de boosting utilise un encodage cible avec validation croisée pour gérer les codes géographiques nombreux.

In [ ]:
import json
from pathlib import Path

METRICS_PATH = Path("ml/evaluation/baseline_metrics.json")
if not METRICS_PATH.exists():
    raise FileNotFoundError(
        "Lance d’abord python ml/training/train_baseline.py"
    )
results = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
print("Découpage :", results["temporal_split"])
print("Effectifs :", results["row_counts"])
print("Variables :", results["features"])

## Résultats sur le dernier trimestre

- **MAE** : erreur absolue moyenne en euros, facile à interpréter.
- **Erreur absolue médiane** : erreur typique, moins sensible aux prix extrêmes.
- **RMSE** : pénalise fortement les grosses erreurs.
- **R²** : part de la variance expliquée par rapport à une prédiction constante. Une valeur négative signifie que le modèle fait moins bien que cette référence sur ce critère.

In [ ]:
import pandas as pd

overall = pd.DataFrame(results["metrics"]).T
by_type = pd.DataFrame(results["ridge_metrics_by_property_type"]).T
hgb_by_type = pd.DataFrame(results["hgb_metrics_by_property_type"]).T
display(overall.round(3))
display(by_type.round(3))
display(hgb_by_type.round(3))
price_bands = pd.DataFrame(
    results["ridge_metrics_by_actual_price_band"]
).T
display(price_bands.round(0))
hgb_price_bands = pd.DataFrame(
    results["hgb_metrics_by_actual_price_band"]
).T
display(hgb_price_bands.round(0))
print("Part de catégories géographiques inconnues au test :")
display(pd.Series(results["unseen_category_share_in_holdout"]))


## Comparaison et interprétation

Sur le dernier trimestre, Ridge obtient une MAE d’environ 69 343 € et un R² de 0,387. Histogram Gradient Boosting obtient une MAE d’environ 71 912 € et un R² de 0,352. Les deux dépassent largement la médiane par type (MAE d’environ 125 069 €). Ridge gagne sur les résultats globaux, mais le boosting fait légèrement mieux sur certaines tranches intermédiaires.

Le quintile de prix supérieur reste difficile pour les deux modèles : la MAE est d’environ 170 816 € pour Ridge et 182 339 € pour le boosting ; leurs erreurs moyennes sont négatives, signe de sous-estimation. La MAE des appartements est plus faible que celle des maisons pour les deux modèles.

Cette comparaison porte sur un seul découpage, une seule année et une cohorte construite avec une clé approximative. Elle ne suffit pas à sélectionner un modèle de production ni à garantir la performance des années futures.

## Suite proposée

Les prix extrêmes ne sont pas supprimés automatiquement : plusieurs hauts montants correspondent à des surfaces bâties ou des parcelles très grandes. La prochaine étape consiste à quantifier les erreurs par tranche de prix, puis à vérifier si la surface de terrain peut être agrégée sans double comptage. Une fois cette variable auditée, comparer à nouveau le même modèle et le même découpage temporel.

## Artefact retenu entraîné sur toute l’année

Après comparaison, relancer `python ml/training/train_baseline.py --refit-ridge-on-full-year` pour enregistrer le Ridge réentraîné sur les 735 202 lignes de 2025 dans `ml/models/ridge_2025_full.joblib`. Le test d’octobre à décembre a servi à choisir le modèle et entre maintenant dans son entraînement : les métriques temporelles ci-dessus concernent le modèle évalué avant ce réentraînement et ne sont pas un score indépendant de l’artefact complet. Ses caractéristiques d’entraînement sont décrites dans `ml/evaluation/ridge_2025_full_metadata.json`.